# Improved Solution - Datathon 2026

Notebook ini memberikan solusi lengkap dengan:
1. Advanced feature engineering
2. SMOTE untuk handle class imbalance
3. Multiple model alternatives
4. Ensemble methods
5. Extensive hyperparameter tuning
6. Feature selection
7. Submission generation

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import SelectKBest, f_classif, RFE
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from imblearn.over_sampling import SMOTE
from scipy.stats import uniform, randint
import warnings
warnings.filterwarnings('ignore')

# Load data
train = pd.read_csv('../data/train.csv')
test = pd.read_csv('../data/test.csv')
sample_sub = pd.read_csv('../data/sample_submission.csv')

print(f"Train shape: {train.shape}")
print(f"Test shape: {test.shape}")
print(f"\nTarget distribution:")
print(train['target'].value_counts())

## 1. Advanced Feature Engineering

In [ ]:
def create_features(df):
    """
    Buat fitur tambahan dari data mentah dengan advanced feature engineering
    """
    df = df.copy()
    
    # 1. Statistik nilai mingguan
    nilai_cols = [col for col in df.columns if 'nilai_minggu' in col]
    df['nilai_mean'] = df[nilai_cols].mean(axis=1)
    df['nilai_std'] = df[nilai_cols].std(axis=1)
    df['nilai_max'] = df[nilai_cols].max(axis=1)
    df['nilai_min'] = df[nilai_cols].min(axis=1)
    df['nilai_range'] = df['nilai_max'] - df['nilai_min']
    
    # 2. Trend nilai (naik/turun)
    df['nilai_trend'] = df[nilai_cols].iloc[:, -1] - df[nilai_cols].iloc[:, 0]
    
    # 3. Advanced Rolling Statistics
    df['nilai_rolling_mean_3'] = df[nilai_cols].iloc[:, -3:].mean(axis=1)
    df['nilai_rolling_std_3'] = df[nilai_cols].iloc[:, -3:].std(axis=1)
    df['nilai_slope'] = df[nilai_cols].iloc[:, -1] - df[nilai_cols].iloc[:, 0]
    df['nilai_volatility'] = df[nilai_cols].std(axis=1)
    df['nilai_75th'] = df[nilai_cols].quantile(0.75, axis=1)
    df['nilai_25th'] = df[nilai_cols].quantile(0.25, axis=1)
    df['nilai_iqr'] = df['nilai_75th'] - df['nilai_25th']
    
    # 4. Statistik aktivitas harian
    aktivitas_cols = [col for col in df.columns if 'aktivitas_hari' in col]
    df['aktivitas_mean'] = df[aktivitas_cols].mean(axis=1)
    df['aktivitas_std'] = df[aktivitas_cols].std(axis=1)
    df['aktivitas_max'] = df[aktivitas_cols].max(axis=1)
    df['aktivitas_min'] = df[aktivitas_cols].min(axis=1)
    df['aktivitas_range'] = df['aktivitas_max'] - df['aktivitas_min']
    
    # Rolling statistics for aktivitas
    df['aktivitas_rolling_mean_3'] = df[aktivitas_cols].iloc[:, -3:].mean(axis=1)
    df['aktivitas_rolling_std_3'] = df[aktivitas_cols].iloc[:, -3:].std(axis=1)
    
    # 5. Rasio tugas
    df['rasio_tugas'] = df['tugas_selesai'] / df['tugas_diberikan']
    df['rasio_tugas'] = df['rasio_tugas'].fillna(0)
    
    # 6. Kombinasi fitur skor
    df['total_skor'] = df['skor_motivasi'] + df['skor_kedisiplinan'] + df['skor_tryout'] + df['skor_literasi']
    df['motivasi_kedisiplinan'] = df['skor_motivasi'] * df['skor_kedisiplinan']
    df['motivasi_tryout'] = df['skor_motivasi'] * df['skor_tryout']
    df['kedisiplinan_literasi'] = df['skor_kedisiplinan'] * df['skor_literasi']
    
    # 7. Fitur interaksi lain
    df['kehadiran_minat'] = df['indeks_kehadiran'] * df['skor_minat_belajar']
    df['tryout_ekstra'] = df['skor_tryout'] * df['skor_ekstrakurikuler']
    df['motivasi_ekstra'] = df['skor_motivasi'] * df['skor_ekstrakurikuler']
    
    # 8. Polynomial features untuk fitur penting
    df['nilai_mean_squared'] = df['nilai_mean'] ** 2
    df['total_skor_squared'] = df['total_skor'] ** 2
    df['rasio_tugas_squared'] = df['rasio_tugas'] ** 2
    
    # 9. Domain-Specific Features (NEW)
    # Consistency metrics
    df['nilai_consistency'] = 1 - (df['nilai_std'] / (df['nilai_mean'].abs() + 1e-6))
    
    # Improvement rate
    df['nilai_improvement_rate'] = (df[nilai_cols].iloc[:, -1] - df[nilai_cols].iloc[:, 0]) / 12
    
    # Activity engagement score
    df['activity_engagement'] = df[aktivitas_cols].gt(df[aktivitas_cols].mean(axis=1)).sum(axis=1) / len(aktivitas_cols)
    
    # Performance trend (positive/negative)
    df['performance_trend'] = np.where(df['nilai_trend'] > 0, 1, 0)
    
    # Task completion efficiency
    df['task_efficiency'] = df['rasio_tugas'] * df['aktivitas_mean']
    
    # Academic potential score
    df['academic_potential'] = (
        df['skor_tryout'] * 0.4 + 
        df['total_skor'] * 0.3 + 
        df['nilai_mean'] * 0.3
    )
    
    # Motivation consistency
    df['motivation_consistency'] = df['skor_motivasi'] * df['nilai_consistency']
    
    # Learning momentum
    df['learning_momentum'] = df['nilai_rolling_mean_3'] * df['aktivitas_rolling_mean_3']
    
    return df

# Apply feature engineering
train_fe = create_features(train)
test_fe = create_features(test)

print(f"Features setelah engineering: {train_fe.shape}")

## 2. Split Data dan Handle Class Imbalance

In [ ]:
# Tentukan fitur yang akan digunakan
drop_cols = ['id', 'target']
feature_cols = [col for col in train_fe.columns if col not in drop_cols]

X = train_fe[feature_cols]
y = train_fe['target']
X_test = test_fe[feature_cols]

# Split data dengan stratifikasi
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape}")
print(f"Validation set: {X_val.shape}")

# Handle Class Imbalance dengan SMOTE
print("\nApplying SMOTE to handle class imbalance...")
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print(f"Original training set shape: {X_train.shape}")
print(f"After SMOTE training set shape: {X_train_smote.shape}")
print(f"\nOriginal target distribution:")
print(pd.Series(y_train).value_counts())
print(f"\nAfter SMOTE target distribution:")
print(pd.Series(y_train_smote).value_counts())

## 3. Training Multiple Models Alternatif

In [ ]:
print("=== Training Multiple Models ===\n")

# 1. Random Forest
rf = RandomForestClassifier(n_estimators=300, max_depth=15, random_state=42, n_jobs=-1)
rf.fit(X_train_smote, y_train_smote)
rf_pred = rf.predict(X_val)
rf_acc = accuracy_score(y_val, rf_pred)
print(f"Random Forest Validation Accuracy: {rf_acc:.4f}")

# 2. LightGBM
lgbm = LGBMClassifier(
    n_estimators=300,
    max_depth=8,
    learning_rate=0.1,
    random_state=42,
    verbose=-1
)
lgbm.fit(X_train_smote, y_train_smote)
lgbm_pred = lgbm.predict(X_val)
lgbm_acc = accuracy_score(y_val, lgbm_pred)
print(f"LightGBM Validation Accuracy: {lgbm_acc:.4f}")

# 3. CatBoost
cat = CatBoostClassifier(
    iterations=300,
    depth=8,
    learning_rate=0.1,
    random_state=42,
    verbose=False
)
cat.fit(X_train_smote, y_train_smote)
cat_pred = cat.predict(X_val)
cat_acc = accuracy_score(y_val, cat_pred)
print(f"CatBoost Validation Accuracy: {cat_acc:.4f}")

# 4. Gradient Boosting
gb = GradientBoostingClassifier(
    n_estimators=300,
    max_depth=8,
    learning_rate=0.1,
    random_state=42
)
gb.fit(X_train_smote, y_train_smote)
gb_pred = gb.predict(X_val)
gb_acc = accuracy_score(y_val, gb_pred)
print(f"Gradient Boosting Validation Accuracy: {gb_acc:.4f}")

print(f"\nBest single model so far: {max(rf_acc, lgbm_acc, cat_acc, gb_acc):.4f}")

## 4. Ensemble Methods

In [ ]:
print("=== Ensemble Methods ===\n")

# 1. Voting Ensemble (Soft Voting)
voting_clf = VotingClassifier(
    estimators=[
        ('xgb', XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1, random_state=42, use_label_encoder=False, eval_metric='mlogloss')),
        ('rf', RandomForestClassifier(n_estimators=300, max_depth=15, random_state=42, n_jobs=-1)),
        ('lgbm', LGBMClassifier(n_estimators=300, max_depth=8, learning_rate=0.1, random_state=42, verbose=-1)),
        ('cat', CatBoostClassifier(iterations=300, depth=8, learning_rate=0.1, random_state=42, verbose=False))
    ],
    voting='soft',
    n_jobs=-1
)

print("Training Voting Classifier...")
voting_clf.fit(X_train_smote, y_train_smote)
voting_pred = voting_clf.predict(X_val)
voting_acc = accuracy_score(y_val, voting_pred)
print(f"Voting Classifier Validation Accuracy: {voting_acc:.4f}")

# 2. Stacking Classifier
stacking_clf = StackingClassifier(
    estimators=[
        ('xgb', XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1, random_state=42, use_label_encoder=False, eval_metric='mlogloss')),
        ('rf', RandomForestClassifier(n_estimators=300, max_depth=15, random_state=42, n_jobs=-1)),
        ('lgbm', LGBMClassifier(n_estimators=300, max_depth=8, learning_rate=0.1, random_state=42, verbose=-1))
    ],
    final_estimator=LogisticRegression(max_iter=1000, random_state=42),
    cv=3,
    n_jobs=-1
)

print("\nTraining Stacking Classifier...")
stacking_clf.fit(X_train_smote, y_train_smote)
stacking_pred = stacking_clf.predict(X_val)
stacking_acc = accuracy_score(y_val, stacking_pred)
print(f"Stacking Classifier Validation Accuracy: {stacking_acc:.4f}")

print(f"\nBest ensemble method: {max(voting_acc, stacking_acc):.4f}")

## 5. CatBoost-Specific Hyperparameter Tuning

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import uniform, randint
from catboost import CatBoostClassifier
import time

print("=== CatBoost-Specific Hyperparameter Tuning ===\n")

# Pastikan X_train_smote dan y_train_smote sudah terdefinisi
# Jika belum, run cell 5 terlebih dahulu

try:
    X_train_smote
    y_train_smote
    print("X_train_smote dan y_train_smote sudah terdefinisi")
except NameError:
    print("ERROR: X_train_smote dan y_train_smote belum terdefinisi!")
    print("Silakan run cell 5 (Split Data dan Handle Class Imbalance) terlebih dahulu.")
    print("Atau run semua cell dari awal (Cell 1, 3, 5) secara berurutan.")
    raise

# === STRATEGI 1: Coarse Search (Pencarian Kasar) ===
print("Stage 1: Coarse Search (Mencari area terbaik)...")
start_time = time.time()

# Parameter distribution yang lebih luas untuk coarse search
param_dist_cat_coarse = {
    'iterations': randint(100, 500),      # Range lebih luas
    'depth': randint(3, 10),
    'learning_rate': uniform(0.01, 0.3),
    'l2_leaf_reg': uniform(1, 10),
    'border_count': randint(32, 200),
    'bagging_temperature': uniform(0, 1),
    'random_strength': uniform(0, 1),
    'one_hot_max_size': randint(2, 10)
}

cat_search_coarse = RandomizedSearchCV(
    CatBoostClassifier(random_state=42, verbose=False),
    param_distributions=param_dist_cat_coarse,
    n_iter=20,          # Dikurangi dari 50
    cv=3,               # Dikurangi dari 5
    scoring='accuracy',
    random_state=42,
    n_jobs=-1,          # Gunakan semua CPU core
    verbose=1
)

print("Melakukan coarse search...")
cat_search_coarse.fit(X_train_smote, y_train_smote)

coarse_time = time.time() - start_time
print(f"\nCoarse search selesai dalam {coarse_time:.2f} detik")
print(f"Best coarse parameters: {cat_search_coarse.best_params_}")
print(f"Best coarse CV accuracy: {cat_search_coarse.best_score_:.4f}")

# === STRATEGI 2: Fine Search (Pencarian Halus) ===
print("\n" + "="*50)
print("Stage 2: Fine Search (Memperhalus parameter)...")
start_time = time.time()

# Ambil best parameters dari coarse search
best_coarse = cat_search_coarse.best_params_

# Buat parameter distribution yang lebih spesifik di sekitar best parameters
param_dist_cat_fine = {
    'iterations': randint(
        max(50, best_coarse['iterations'] - 100), 
        best_coarse['iterations'] + 100
    ),
    'depth': randint(
        max(2, best_coarse['depth'] - 2), 
        min(12, best_coarse['depth'] + 2)
    ),
    'learning_rate': uniform(
        max(0.005, best_coarse['learning_rate'] - 0.05),
        min(0.1, best_coarse['learning_rate'] + 0.1)
    ),
    'l2_leaf_reg': uniform(
        max(0.5, best_coarse['l2_leaf_reg'] - 2),
        min(5, best_coarse['l2_leaf_reg'] + 2)
    ),
    'border_count': randint(
        max(16, best_coarse['border_count'] - 50),
        min(255, best_coarse['border_count'] + 50)
    ),
    'bagging_temperature': uniform(
        max(0, best_coarse['bagging_temperature'] - 0.2),
        min(0.4, best_coarse['bagging_temperature'] + 0.2)
    ),
    'random_strength': uniform(
        max(0, best_coarse['random_strength'] - 0.2),
        min(0.4, best_coarse['random_strength'] + 0.2)
    ),
    'one_hot_max_size': randint(
        max(1, best_coarse['one_hot_max_size'] - 3),
        min(15, best_coarse['one_hot_max_size'] + 3)
    )
}

cat_search_fine = RandomizedSearchCV(
    CatBoostClassifier(random_state=42, verbose=False),
    param_distributions=param_dist_cat_fine,
    n_iter=15,          # Lebih sedikit untuk fine-tuning
    cv=5,               # Kembali ke 5 folds untuk hasil lebih akurat
    scoring='accuracy',
    random_state=42,
    n_jobs=-1,
    verbose=1
)

print("Melakukan fine search...")
cat_search_fine.fit(X_train_smote, y_train_smote)

fine_time = time.time() - start_time
print(f"\nFine search selesai dalam {fine_time:.2f} detik")
print(f"Best fine parameters: {cat_search_fine.best_params_}")
print(f"Best fine CV accuracy: {cat_search_fine.best_score_:.4f}")

# === EVALUASI FINAL ===
print("\n" + "="*50)
print("=== Evaluasi Final ===")

# Pilih model terbaik dari fine search
best_cat_model = cat_search_fine.best_estimator_

# Evaluasi pada validation set
cat_tuned_pred = best_cat_model.predict(X_val)
cat_tuned_acc = accuracy_score(y_val, cat_tuned_pred)

print(f"Validation Accuracy dengan tuned CatBoost: {cat_tuned_acc:.4f}")

# === PERBANDINGAN WAKTU ===
total_time = coarse_time + fine_time
print(f"\nTotal waktu tuning: {total_time:.2f} detik ({total_time/60:.2f} menit)")

# Estimasi waktu jika menggunakan 50 candidates × 5 folds
estimated_old_time = (total_time / (20*3 + 15*5)) * (50*5)
print(f"Estimasi waktu dengan metode lama (50 candidates × 5 folds): {estimated_old_time:.2f} detik ({estimated_old_time/60:.2f} menit)")
print(f"Penghematan waktu: {(1 - total_time/estimated_old_time)*100:.1f}%")

## 6. Robust Cross-Validation dengan Repeated Stratified K-Fold

In [ ]:
print("=== Robust Cross-Validation dengan Repeated Stratified K-Fold ===\n")

from sklearn.model_selection import RepeatedStratifiedKFold

# Gunakan Repeated Stratified K-Fold untuk mengurangi variance
rskf = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=42)

# Gunakan CatBoost sederhana untuk cross-validation (tanpa parameter yang menyebabkan cloning error)
cat_for_cv = CatBoostClassifier(
    iterations=300,
    depth=8,
    learning_rate=0.1,
    random_state=42,
    verbose=False
)

print("Evaluasi CatBoost dengan robust CV...")
cat_cv_scores = cross_val_score(
    cat_for_cv, 
    X_train_smote, 
    y_train_smote, 
    cv=rskf, 
    scoring='accuracy',
    n_jobs=-1
)

print(f"CatBoost Repeated Stratified K-Fold CV:")
print(f"Mean CV: {cat_cv_scores.mean():.4f} (+/- {cat_cv_scores.std():.4f})")
print(f"Min CV: {cat_cv_scores.min():.4f}")
print(f"Max CV: {cat_cv_scores.max():.4f}")

# Evaluasi LightGBM dengan robust CV (model terbaik kedua)
lgbm_for_cv = LGBMClassifier(n_estimators=300, max_depth=8, learning_rate=0.1, random_state=42, verbose=-1)
lgbm_cv_scores = cross_val_score(
    lgbm_for_cv,
    X_train_smote,
    y_train_smote,
    cv=rskf,
    scoring='accuracy',
    n_jobs=-1
)

print(f"\nLightGBM Repeated Stratified K-Fold CV:")
print(f"Mean CV: {lgbm_cv_scores.mean():.4f} (+/- {lgbm_cv_scores.std():.4f})")

print(f"\nCatatan: Cross-validation menggunakan model CatBoost sederhana untuk menghindari cloning error.")
print(f"Model CatBoost yang di-tune (best_cat_model) akan digunakan untuk final submission.")

## 7. Weighted Ensemble dengan Top 2 Models (CatBoost + LightGBM)

In [ ]:
print("=== Weighted Ensemble dengan Top 2 Models ===\n")

# Gunakan CatBoost sederhana untuk ensemble (untuk menghindari cloning error)
cat_simple = CatBoostClassifier(
    iterations=300,
    depth=8,
    learning_rate=0.1,
    random_state=42,
    verbose=False
)

# Ensemble CatBoost + LightGBM dengan weighted voting
voting_top2 = VotingClassifier(
    estimators=[
        ('cat', cat_simple),  # CatBoost sederhana
        ('lgbm', LGBMClassifier(n_estimators=300, max_depth=8, learning_rate=0.1, random_state=42, verbose=-1))
    ],
    voting='soft',
    weights=[0.6, 0.4],  # Beri lebih banyak weight ke CatBoost
    n_jobs=-1
)

print("Training Weighted Voting Classifier (CatBoost + LightGBM)...")
voting_top2.fit(X_train_smote, y_train_smote)
voting_top2_pred = voting_top2.predict(X_val)
voting_top2_acc = accuracy_score(y_val, voting_top2_pred)
print(f"Weighted Voting Validation Accuracy: {voting_top2_acc:.4f}")

# Coba juga dengan weights berbeda
for w1, w2 in [(0.7, 0.3), (0.5, 0.5), (0.8, 0.2)]:
    voting_test = VotingClassifier(
        estimators=[
            ('cat', cat_simple),
            ('lgbm', LGBMClassifier(n_estimators=300, max_depth=8, learning_rate=0.1, random_state=42, verbose=-1))
        ],
        voting='soft',
        weights=[w1, w2],
        n_jobs=-1
    )
    voting_test.fit(X_train_smote, y_train_smote)
    voting_test_pred = voting_test.predict(X_val)
    voting_test_acc = accuracy_score(y_val, voting_test_pred)
    print(f"Weighted Voting (CatBoost={w1}, LightGBM={w2}): {voting_test_acc:.4f}")

print(f"\nCatatan: Ensemble menggunakan CatBoost sederhana untuk menghindari cloning error.")
print(f"Model CatBoost yang di-tune (best_cat_model) akan digunakan untuk final submission.")

In [ ]:
print("=== Calibration untuk Post-Processing ===\n")

from sklearn.calibration import CalibratedClassifierCV

# Gunakan CatBoost sederhana untuk calibration (untuk menghindari cloning error)
cat_simple = CatBoostClassifier(
    iterations=300,
    depth=8,
    learning_rate=0.1,
    random_state=42,
    verbose=False
)

# Calibrate CatBoost probabilities untuk improve prediction
calibrated_cat = CalibratedClassifierCV(
    cat_simple, 
    method='isotonic', 
    cv=5
)

print("Training Calibrated CatBoost...")
calibrated_cat.fit(X_train_smote, y_train_smote)
calibrated_cat_pred = calibrated_cat.predict(X_val)
calibrated_cat_acc = accuracy_score(y_val, calibrated_cat_pred)
print(f"Calibrated CatBoost Validation Accuracy: {calibrated_cat_acc:.4f}")

# Coba juga dengan sigmoid calibration
calibrated_cat_sigmoid = CalibratedClassifierCV(
    cat_simple,
    method='sigmoid',
    cv=5
)

print("\nTraining Calibrated CatBoost (Sigmoid)...")
calibrated_cat_sigmoid.fit(X_train_smote, y_train_smote)
calibrated_cat_sigmoid_pred = calibrated_cat_sigmoid.predict(X_val)
calibrated_cat_sigmoid_acc = accuracy_score(y_val, calibrated_cat_sigmoid_pred)
print(f"Calibrated CatBoost (Sigmoid) Validation Accuracy: {calibrated_cat_sigmoid_acc:.4f}")

print(f"\nCatatan: Calibration menggunakan CatBoost sederhana untuk menghindari cloning error.")

## Summary

Notebook ini telah menerapkan strategi improve dari baseline 49% ke target 70-80%:

### Improvements yang Diterapkan:
1. **Advanced Feature Engineering** - rolling statistics, trend analysis, polynomial features
2. **Domain-Specific Features** - consistency metrics, improvement rate, academic potential score, learning momentum
3. **SMOTE** - untuk handle class imbalance
4. **Multiple Model Alternatives** - Random Forest, LightGBM, CatBoost, Gradient Boosting
5. **CatBoost-Specific Hyperparameter Tuning** - RandomizedSearchCV dengan 50 iterasi
6. **Robust Cross-Validation** - Repeated Stratified K-Fold (5 splits, 3 repeats)
7. **Weighted Ensemble** - Top 2 models (CatBoost + LightGBM) dengan weighted voting
8. **Calibration** - Isotonic dan sigmoid calibration untuk post-processing

### Hasil Eksperimen (sebelum improvement):
- Baseline: 49.38%
- CatBoost single: 53.75%
- Voting Classifier: 51.25%

### Target Improvement:
- **Target**: 70-80% (dengan advanced feature engineering + ensemble + optimal strategy)

### Submission Files:
- `outputs/submission_final.csv` - CatBoost tuned (recommended)
- `outputs/submission_all_strategies.csv` - Semua model predictions untuk comparison

### Catatan Penting:
- Feature selection TIDAK digunakan karena menurunkan akurasi
- Fokus pada CatBoost karena sudah terbukti terbaik
- Ensemble hanya efektif dengan 2 model terbaik
- Robust CV penting untuk menghindari overfitting

In [ ]:
print("=== Generate Final Submission dengan Best Blending ===\n")

# Apply SMOTE pada full training data
smote_full = SMOTE(random_state=42)
X_full_smote, y_full_smote = smote_full.fit_resample(X, y)

# Train CatBoost dengan best tuned parameters pada full dataset
print("Training CatBoost with best tuned parameters on full dataset...")
cat_final = CatBoostClassifier(
    iterations=436,
    depth=5,
    learning_rate=0.16015757296260286,
    l2_leaf_reg=11.734628978618437,
    border_count=228,
    bagging_temperature=0.06423578631931819,
    random_strength=0.33169390030136725,
    one_hot_max_size=2,
    random_state=42,
    verbose=False
)
cat_final.fit(X_full_smote, y_full_smote)

# Train LightGBM pada full dataset
print("Training LightGBM on full dataset...")
lgbm_final = LGBMClassifier(n_estimators=300, max_depth=8, learning_rate=0.1, random_state=42, verbose=-1)
lgbm_final.fit(X_full_smote, y_full_smote)

# Get probabilities for test set
cat_proba_test = cat_final.predict_proba(X_test)
lgbm_proba_test = lgbm_final.predict_proba(X_test)

# Use best weights dari validation (ganti dengan hasil terbaik dari cell sebelumnya)
# Default 0.7, 0.3 jika belum run cell sebelumnya
best_w_cat, best_w_lgbm = 0.7, 0.3

# Blending
blended_proba_test = best_w_cat * cat_proba_test + best_w_lgbm * lgbm_proba_test
test_predictions = np.argmax(blended_proba_test, axis=1)

# Buat submission file
submission = pd.DataFrame({
    'id': test['id'],
    'target': test_predictions
})

submission.to_csv('outputs/submission_final.csv', index=False)
print("Submission file saved as 'outputs/submission_final.csv'")
print(f"\nSubmission shape: {submission.shape}")
print(submission.head())

# Simpan juga individual predictions untuk comparison
submission['target_catboost'] = np.argmax(cat_proba_test, axis=1)
submission['target_lgbm'] = np.argmax(lgbm_proba_test, axis=1)
submission.to_csv('outputs/submission_all_strategies.csv', index=False)
print("\nAll strategy predictions saved as 'outputs/submission_all_strategies.csv'")

print(f"\n=== Summary ===")
print(f"Submission strategy: Blending (CatBoost={best_w_cat}, LightGBM={best_w_lgbm})")
print(f"Files generated:")
print("1. outputs/submission_final.csv - Best blending (recommended)")
print("2. outputs/submission_all_strategies.csv - Individual predictions for comparison")

In [ ]:
print("=== Manual Ensemble dengan Best Tuned Parameters ===\n")

# Gunakan parameter terbaik dari tuning untuk buat CatBoost baru
# Ini menghindari cloning error karena kita buat instance baru, bukan clone dari best_cat_model
cat_manual = CatBoostClassifier(
    iterations=436,
    depth=5,
    learning_rate=0.16015757296260286,
    l2_leaf_reg=11.734628978618437,
    border_count=228,
    bagging_temperature=0.06423578631931819,
    random_strength=0.33169390030136725,
    one_hot_max_size=2,
    random_state=42,
    verbose=False
)

print("Training CatBoost dengan best tuned parameters...")
cat_manual.fit(X_train_smote, y_train_smote)
cat_manual_pred = cat_manual.predict(X_val)
cat_manual_acc = accuracy_score(y_val, cat_manual_pred)
print(f"CatBoost (Manual Tuned) Validation Accuracy: {cat_manual_acc:.4f}")

# Train LightGBM juga
lgbm_manual = LGBMClassifier(n_estimators=300, max_depth=8, learning_rate=0.1, random_state=42, verbose=-1)
print("\nTraining LightGBM...")
lgbm_manual.fit(X_train_smote, y_train_smote)
lgbm_manual_pred = lgbm_manual.predict(X_val)
lgbm_manual_acc = accuracy_score(y_val, lgbm_manual_pred)
print(f"LightGBM Validation Accuracy: {lgbm_manual_acc:.4f}")

# Manual blending dengan weighted average of probabilities
print("\n=== Manual Blending (Weighted Average of Probabilities) ===")

# Get probabilities
cat_proba = cat_manual.predict_proba(X_val)
lgbm_proba = lgbm_manual.predict_proba(X_val)

# Coba beberapa weight combinations
weights = [(0.7, 0.3), (0.6, 0.4), (0.5, 0.5), (0.8, 0.2), (0.75, 0.25)]

best_blend_acc = 0
best_weights = None

for w_cat, w_lgbm in weights:
    # Weighted average of probabilities
    blended_proba = w_cat * cat_proba + w_lgbm * lgbm_proba
    blended_pred = np.argmax(blended_proba, axis=1)
    blended_acc = accuracy_score(y_val, blended_pred)
    print(f"Blending (CatBoost={w_cat}, LightGBM={w_lgbm}): {blended_acc:.4f}")
    
    if blended_acc > best_blend_acc:
        best_blend_acc = blended_acc
        best_weights = (w_cat, w_lgbm)

print(f"\nBest blending: {best_blend_acc:.4f} with weights CatBoost={best_weights[0]}, LightGBM={best_weights[1]}")

# Compare dengan individual models
print(f"\n=== Comparison ===")
print(f"CatBoost Manual Tuned: {cat_manual_acc:.4f}")
print(f"LightGBM: {lgbm_manual_acc:.4f}")
print(f"Best Blending: {best_blend_acc:.4f}")
print(f"Improvement dari CatBoost: {(best_blend_acc - cat_manual_acc)*100:.2f}%")